# لاب ٣ على Google Colab — BAYAN.DAICO
هذا الدفتر يشغّل كود المشروع نفسه؛ لا يحتوي تنفيذًا ثانيًا للنماذج.

**قبل البداية:** اختر **Runtime → Change runtime type**، واضبط **Runtime Version = 2026.07** و**Hardware accelerator = T4 GPU** ثم احفظ وأعد الاتصال. إصدار 2026.07 يوفر Python 3.12.13 المتوافق مع المشروع. لا تتابع الخلايا إذا أخفق فحص البيئة.
التدريب المحلي مكتمل. هذه تجربة إعادة إنتاج على GPU، وليست وسيلة لتعديل النماذج بعد الاطلاع على الاختبار المجمد.

اختبار الموضوع يحتوي ٤ فئات فقط من ٨. تحديث المقرر بتاريخ المزامنة 2026-09-09 يطلب في QA إجابة صحيحة على الأسئلة الـ١٢؛ الملف لا يحتوي أسئلة بلا إجابة، لذلك نحتفظ بتجربة 9+3 الإضافية لفحصها. تفاصيل ذلك في `docs/LAB3_WALKTHROUGH.md`.

شغّل الخلايا بالترتيب بزر ▶. اختر **Runtime → Change runtime type → T4 GPU** أو GPU متاح. المشروع يحتاج Python 3.12؛ إذا اختلف الإصدار استخدم إصدار تشغيل متاحًا يدعم 3.12: https://research.google.com/colaboratory/runtime-version-faq.html


In [ ]:
import sys, subprocess
RUNTIME_READY = False
print("Python:", sys.version.split()[0])
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "Python 3.12 is required. In Colab: Runtime > Change runtime type > "
        "Runtime Version: 2026.07; Hardware accelerator: T4 GPU. "
        "Save, reconnect, and rerun this cell before continuing."
    )
subprocess.run([
    sys.executable, "-c",
    "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable'); "
    "assert torch.cuda.is_available(), 'Select T4 GPU in Runtime > Change runtime type'"
], check=True)
RUNTIME_READY = True
print("Runtime ready: Python 3.12 + CUDA GPU")


## ١. جلب نسخة المشروع أو تحديث النسخة القديمة
هذه الخلية تجلب المستودع عند أول تشغيل. إذا كان موجودًا من جلسة سابقة، تجلب آخر تحديث وتدمجه بـfast-forward فقط. بذلك تصل تعديلات لاب ٣ بدل البقاء على commit لاب ٢.
إذا منع Git الدمج بسبب تعديلات محلية، احتفظ بها أولًا؛ الخلية لا تحذفها أو تستخدم force.


In [ ]:
assert globals().get("RUNTIME_READY", False), "Complete the Python 3.12 / GPU setup cell first."
REPO_READY = False
from pathlib import Path
import os
REPO_URL = "https://github.com/jnjnmaizi/BAYAN.DAICO.git"
PROJECT = Path("/content/BAYAN.DAICO")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
else:
    assert (PROJECT / ".git").is_dir(), "Existing project folder is not a Git checkout; preserve it before using a fresh runtime."
    remote = subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip()
    assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Existing folder points to a different repository."
    branch = subprocess.check_output(["git", "-C", str(PROJECT), "branch", "--show-current"], text=True).strip()
    assert branch == "main", "Preserve your branch changes before switching to the course main branch."
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "merge", "--ff-only", "FETCH_HEAD"], check=True)
os.chdir(PROJECT)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True))
assert (PROJECT / "src/bayan/models/training.py").exists(), "Lab 3 code is missing from this checkout. Check the repository version shown above."
REPO_READY = True


## ٢. تثبيت مكتبات المشروع
نثبت المتطلبات من الملف نفسه. ننفّذ التدريب لاحقًا في عمليات Python جديدة، حتى لا نعتمد على مكتبات سبق تحميلها داخل نواة الدفتر.


In [ ]:
assert globals().get("RUNTIME_READY", False) and globals().get("REPO_READY", False), "Complete runtime and repository setup before installing."
# This project uses PyTorch; do not load Colab's unrelated TensorFlow/Keras stack.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "scripts/doctor.py"], check=True)


## ٣. ربط Google Drive وحفظ النتائج
وافق على ربط حسابك عندما تظهر نافذة Google. حدد اسم تجربة واحدًا واستخدمه في جميع الخطوات؛ لا تغيّره لمجرد تجاوز حارس إعادة الاختبار.
نحفظ النماذج في Drive لأن ملفات `/content` مؤقتة. ملفات النماذج كبيرة، فتأكد أن مساحة Drive كافية.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "lab3_colab_01"
RUN_DIR = Path("/content/drive/MyDrive/BAYAN.DAICO") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_DIR = RUN_DIR / "tfidf_baseline"
TOPIC_DIR = RUN_DIR / "topic_classifier"
NER_DIR = RUN_DIR / "ner"
QA_DIR = RUN_DIR / "qa"
print("نتائج هذه التجربة:", RUN_DIR)

import shutil, json
def save_evidence():
    source = PROJECT / "artifacts/lab3"
    if source.exists():
        shutil.copytree(source, RUN_DIR / "reports", dirs_exist_ok=True)
    (RUN_DIR / "source_commit.txt").write_text(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))

def run_script(script, *args):
    subprocess.run([sys.executable, "-u", f"scripts/{script}", *map(str, args)], check=True)
    save_evidence()

# الاحتفاظ بتقارير التشغيل المحلي في مجلد مرجعي منفصل عن نتائج Colab الجديدة.
# يتم النقل داخل clone المؤقت مرة واحدة فقط؛ ملفات Drive لا تتأثر.
reference = PROJECT / "artifacts/lab3_repository_reference"
evidence = PROJECT / "artifacts/lab3"
if not reference.exists() and evidence.exists():
    evidence.rename(reference)
evidence.mkdir(parents=True, exist_ok=True)


## ٤. اختبارات صحة الكود
هذه الخطوة تتحقق من التقسيم، ومحاذاة NER، واختيار مدى الإجابة. لا تدرب النماذج ولا تقيس جودتها.


In [ ]:
import sys, subprocess
assert globals().get("RUNTIME_READY", False) and globals().get("REPO_READY", False), (
    "شغّل خلايا فحص Python وGPU، وجلب المشروع، وتثبيت المكتبات بالترتيب في الجلسة الحالية، ثم أعد تشغيل هذه الخلية."
)
subprocess.run([sys.executable, "-m", "pytest", "tests/test_model_data.py", "tests/test_ner_alignment.py", "tests/test_qa.py", "tests/test_lab3_regressions.py", "-q"], cwd=PROJECT, check=True)


## ٥. المرجع السريع ثم تدريب التصنيف
نقيس المرجع قبل مقارنته بالنموذج الأكبر. تدريب XLM-R يختار أفضل حقبة باستخدام التحقق فقط. يستخدم CUDA تلقائيًا عند توفر GPU. أول تشغيل ينزّل أوزان النموذج.


In [ ]:
run_script("tfidf_baseline.py", "--output-dir", BASELINE_DIR)


In [ ]:
run_script("train_classifier.py", "--output-dir", TOPIC_DIR)


## ٦. المقارنة النهائية للتصنيف — مرة واحدة
شغّلها بعد انتهاء التدريب وتثبيت الاختيار. تستخدم المرجع المحفوظ وتقارن على الاختبار نفسه؛ لا تضبط التدريب استنادًا إلى هذه النتيجة. `--baseline-dir` مهم لأن المرجع محفوظ في Drive.


In [ ]:
run_script("train_classifier.py", "--output-dir", TOPIC_DIR, "--baseline-dir", BASELINE_DIR, "--evaluate-only", "--evaluate-test")


## ٧. تدريب NER ثم اختبار النموذج المختار
نقسّم القوالب المتكررة إلى مجموعات منفصلة، ونختار أفضل حقبة على التحقق. شغّل خلية الاختبار مرة واحدة فقط بعد انتهاء التدريب.


In [ ]:
run_script("train_ner.py", "--output-dir", NER_DIR)


In [ ]:
run_script("train_ner.py", "--output-dir", NER_DIR, "--evaluate-only", "--evaluate-test")


## ٨. QA واختبار عدم الإجابة
نستخدم نموذج QA جاهزًا؛ لا ندرّب نموذجًا جديدًا هنا. نضبط عتبة عدم الإجابة على سياقات تطوير منفصلة، ثم نشغّل الملف الأصلي وتجربة إضافية مستقلة 9+3. الحساب الحالي يعمل على CPU؛ اختيار GPU للدفتر لا يغيّر هذا الجزء.


In [ ]:
run_script("qa_smoke.py", "--output-dir", QA_DIR)


## ٩. عرض النتائج والاحتفاظ بها
انسخ الأرقام من هذه التجربة فقط إذا كان المطلوب تسليم نتائج Colab. قد تختلف الأرقام عن تشغيل الماك؛ لا تغيّرها لتطابق النتائج السابقة. ملفات النماذج في Drive، والتقارير في مجلد `reports`.


In [ ]:
for name, directory in [("TF-IDF", BASELINE_DIR), ("XLM-R topic", TOPIC_DIR), ("NER", NER_DIR), ("QA", QA_DIR)]:
    report = json.loads((directory / "metrics.json").read_text())
    print("\n", name)
    for key in ["validation", "frozen_test", "delta_macro_f1_points", "target_met", "train_seconds", "course_smoke_assessment"]:
        if key in report:
            print(key, report[key])
    for key in ["supplied_smoke", "supplemental_9_plus_3"]:
        if key in report:
            print(key, {k: v for k, v in report[key].items() if k != "predictions"})
save_evidence()
print("تم حفظ التقارير في:", RUN_DIR / "reports")


إذا توقف التشغيل: احتفظ برسالة الخطأ واسم الخلية. لا تغيّر ملف البيانات أو expected answers لمعالجة فشل أحد الأهداف. حراس التقييم سيمنعون إعادة الاختبار في مجلد يحتوي نتائج نهائية؛ استخدم التقارير المحفوظة للمراجعة.

بعد الانتهاء اختر **Runtime → Disconnect and delete runtime**؛ ملفات Drive تبقى محفوظة. إتاحة GPU ومدد الجلسات تعتمد على موارد Colab: https://research.google.com/colaboratory/faq.html
